#  Data Cleaning & Model Training — Villas Vente Marrakech

##  Imports & Configuration

In [1]:
import os, sys, json, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

sys.path.append(os.path.abspath("../../pipeline"))

RAW_PATH   = "../../data/marrakech_immo_vente/villa_vente.csv"
CLEAN_PATH = "../../data/cleaned_data/vente/villa_vente_final.csv"
MODEL_PATH = "../../model_training/models/xgb_villa_vente.pkl"
META_PATH  = "../../model_training/models/xgb_villa_vente_metadata.json"

print(" Imports OK")


 Imports OK


/home/nouhayla/Desktop/stage/gateone-deploy/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement du fichier brut

In [2]:
df = pd.read_csv(RAW_PATH)
print(f"Shape initiale : {df.shape}")
print(f"\nMissing values principales :")
missing = df.isnull().sum()
print(missing[missing > 0])
df.head(3)


Shape initiale : (4729, 34)

Missing values principales :
id                         1686
titre                         9
prix                         55
localisation                  1
type_bien                     1
surface                      65
chambres                    244
salles_bain                1371
description                 953
agence                     1876
url                           1
surface_terrain            4703
prix_num                    599
surface_num                 139
prix_m2                     728
prix_m2_median_quartier      86
dtype: int64


,id,titre,prix,localisation,type_bien,surface,chambres,salles_bain,description,agence,...,etage,surface_terrain,prix_num,surface_num,chambres_num,salles_bain_num,nb_pieces,quartier,prix_m2,prix_m2_median_quartier
0,8163854.0,Villa 4 Suites d’Architecte à Marrakech,690 000 EUR,"Route de l'Ourika, Marrakech",Maison,750 m²,4 Chambres,4 Salles de bains,À seulement 15 minutes du centre-ville de Marr...,Particulier,...,-1,NaN,690000.0,750.0,4.0,4.0,9.0,Route d'Ourika,920.000000,10800.000000
1,7643604.0,Splendide villa à vendre,6 700 000 DH,"Agdal, Marrakech",Maison,500 m²,NaN,NaN,Villa de haut standing à vendre à Agdal. Surfa...,Particulier,...,-1,NaN,6700000.0,500.0,0.0,0.0,1.0,Agdal,13400.000000,12839.024390
2,8230473.0,Maison de haut standing à vendre à Route Amizm...,8 290 000 DH,"Route Amizmiz, Marrakech",Maison,338 m²,4 Chambres,5 Salles de bains,SANCTUARY est une collection confidentielle de...,Particulier,...,-1,NaN,8290000.0,338.0,4.0,5.0,10.0,Route d'Amizmiz,24526.627219,12828.947368


In [3]:
df_raw = pd.read_csv("../../data/marrakech_immo_vente/villa_vente.csv")
print(f"Lignes totales     : {len(df_raw)}")
print(f"id uniques         : {df_raw['id'].nunique()}")
print(f"id dupliqués       : {df_raw['id'].duplicated().sum()}")

# Voir si les doublons sont vraiment identiques
dupes = df_raw[df_raw["id"].duplicated(keep=False)].sort_values("id")
print(f"\nExemple doublons (même id, même prix ?) :")
print(dupes[["id","prix","localisation","surface"]].head(10))

Lignes totales     : 4729
id uniques         : 3035
id dupliqués       : 1693

Exemple doublons (même id, même prix ?) :
            id          prix                            localisation  surface
1625         0  28,889,400Dh  Marrakech, Extérieur, Route Ouarzazate  1500 m²
1709         0  12,000,000Dh      Marrakech, Extérieur, Route Ourika   500 m²
1731         0  20,539,000Dh               Marrakech, Golfs, Amelkis   915 m²
1798         0   8,864,200Dh      Marrakech, Extérieur, Route Ourika   375 m²
1599         1   4,111,600Dh      Marrakech, Extérieur, Route Ourika   200 m²
1710         1  12,000,000Dh      Marrakech, Extérieur, Route Ourika   500 m²
1705  85408439   7,195,300Dh      Marrakech, Extérieur, Route Ourika   323 m²
1855  85408439      66,142Dh                               Marrakech   330 m²
1704  85408569   9,034,700Dh      Marrakech, Extérieur, Route Ourika   323 m²
1799  85408569      77,243Dh                               Marrakech   323 m²


In [4]:
# print(f"Surface terrain extraite du texte : {(df['has_surface_terrain']==1).sum()} / {len(df)}")
# print(f"Valeurs extraites (sample) :")
# print(df[df["has_surface_terrain"]==1][["surface_num","surface_terrain_text","prix_num"]].head(10))

In [5]:
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

#  1. Recharger SANS déduplication sur id 
df = pd.read_csv("../../data/marrakech_immo_vente/villa_vente.csv")
print(f"Brut : {df.shape}")

# Déduplication sur le contenu réel (pas sur id)
df = df.drop_duplicates(subset=["prix_num","surface_num","localisation","chambres_num"]).copy()
print(f"Après dédup contenu : {df.shape}")



Brut : (4729, 34)
Après dédup contenu : (4460, 34)


In [6]:
# ── 2. Cleaning ──────────────────────────────────────────────────────────
EUR_TO_MAD = 10.8
eur_mask = df["prix"].str.contains("EUR", na=False)
df.loc[eur_mask, "prix_num"] = df.loc[eur_mask, "prix_num"] * EUR_TO_MAD
df = df[df["prix_num"].notna() & (df["prix_num"] > 0)].copy()
df = df[df["surface_num"].notna() & (df["surface_num"] >= 10) & (df["surface_num"] <= 3000)].copy()
df["_ppm2"] = df["prix_num"] / df["surface_num"]
df = df[(df["_ppm2"] >= 4_000) & (df["_ppm2"] <= 60_000)].copy()
df.drop(columns=["_ppm2"], inplace=True)
df = df[df["prix_num"] >= 1_000_000].copy()
log_prix = np.log(df["prix_num"])
df = df[(log_prix >= log_prix.quantile(0.01)) & (log_prix <= log_prix.quantile(0.99))].copy()
df["etage"] = df["etage"].replace(-1, np.nan)
df["etage_known"] = df["etage"].notna().astype(int)
df["etage"] = df["etage"].fillna(-99)
df["chambres_num"] = df["chambres_num"].replace(0, np.nan)
df.loc[df["chambres_num"] > 15, "chambres_num"] = np.nan
df["chambres_num"] = df["chambres_num"].fillna(df["chambres_num"].median())
df["salles_bain_num"] = df["salles_bain_num"].fillna(df["salles_bain_num"].median())
print(f"Après cleaning : {df.shape}")



Après cleaning : (3075, 35)


In [7]:
#  3. Features textuelles (uniquement celles avec vraie valeur) 
df["desc"] = df["description"].fillna("").str.lower()

keywords = {
    "kw_standing"    : r"standing|luxe|luxueux|haut de gamme|prestige|exception|premium|raffiné",
    "kw_renove"      : r"rénov|réhabilit|refait|neuf|moderne|contemporain|récent",
    "kw_architecte"  : r"architecte|design|signé|contemporain|concept",
    "kw_jardin"      : r"jardin (aménagé|arboré|paysagé|verdoyant)|parc|espace vert",
    "kw_projet"      : r"projet|sur plan|en cours|livraison|promotion",
}
for feat, pattern in keywords.items():
    df[feat] = df["desc"].str.contains(pattern, regex=True, na=False).astype(int)

df["text_standing_score"] = df[["kw_standing","kw_renove","kw_architecte","kw_jardin"]].sum(axis=1)

# surface_terrain : uniquement les vraies valeurs extraites, NaN sinon
def extract_terrain(text):
    m = re.search(r"(\d[\d\s]{1,6})\s*m.?\s*(de\s+)?(terrain|parcelle|lot)", str(text))
    if m:
        try: return float(m.group(1).replace(" ",""))
        except: return np.nan
    return np.nan

df["surface_terrain_text"] = df["desc"].apply(extract_terrain)
df["has_surface_terrain"]  = df["surface_terrain_text"].notna().astype(int)
# NE PAS imputer le fallback — laisser NaN et laisser XGBoost gérer
df["surface_terrain_text"] = df["surface_terrain_text"]  # NaN = inconnu

df.drop(columns=["desc"], inplace=True)



In [8]:
#  4. Features de base 
top_q = df["quartier"].value_counts().index[:10]
df["quartier_clean"] = df["quartier"].apply(lambda x: x if x in top_q else "Autre")
df["localisation_fine"] = (
    df["localisation"].str.split(",").str[0].str.strip().str.lower()
    .str.replace(r"[^a-zàâäéèêëîïôùûü '\-]", "", regex=True)
)
counts = df["localisation_fine"].value_counts()
df["localisation_fine"] = df["localisation_fine"].apply(
    lambda x: x if x in counts[counts >= 8].index else "autre_zone"
)
df["surface_par_chambre"] = df["surface_num"] / df["chambres_num"].clip(lower=1)
df["score_standing"]      = df["piscine"]+df["hammam"]+df["climatisation"]+df["securite"]+df["vue"]
df["score_exterieur"]     = df["terrasse"]+df["jardin"]+df["piscine"]
df["nb_equipements"]      = df[["piscine","parking","ascenseur","terrasse","jardin",
                                 "climatisation","securite","vue","cave","hammam"]].sum(axis=1)
df["standing_premium"]    = df["piscine"]*3+df["hammam"]*2+df["securite"]+df["vue"]*2+df["climatisation"]
df["ratio_ch_surface"]    = df["chambres_num"] / df["surface_num"].clip(lower=1)
df["cat_surface"] = pd.cut(df["surface_num"],
    bins=[0,200,350,500,800,1500,9999],
    labels=["tiny","small","medium","large","xlarge","estate"]).astype(str)
df["log_prix"] = np.log(df["prix_num"])

print(f"✅ Shape finale : {df.shape}")
print(f"   surface_terrain extrait : {df['has_surface_terrain'].sum()} lignes")



✅ Shape finale : (3075, 53)
   surface_terrain extrait : 98 lignes


In [9]:
# ── 5. Split ─────────────────────────────────────────────────────────────
NUMERIC_FEATURES_BASE = [
    "surface_num","chambres_num","salles_bain_num","etage","etage_known",
    "surface_par_chambre","score_standing","score_exterieur","nb_equipements",
    "standing_premium","ratio_ch_surface",
    "text_standing_score","surface_terrain_text","has_surface_terrain",
]
BINARY_FEATURES = [
    "piscine","parking","ascenseur","terrasse","jardin","climatisation",
    "securite","vue","meuble","neuf","cave","hammam",
    "kw_standing","kw_renove","kw_architecte","kw_jardin","kw_projet",
]
CATEGORICAL_FEATURES = ["quartier_clean","localisation_fine","cat_surface"]

BASE_FEATURES = NUMERIC_FEATURES_BASE + BINARY_FEATURES + CATEGORICAL_FEATURES
X = df[BASE_FEATURES].copy()
y = df["log_prix"].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train : {X_train.shape[0]} | Test : {X_test.shape[0]}")



Train : 2460 | Test : 615


In [10]:
#  6. Groupby features sans leakage 
def add_groupby_features(X_tr, X_te):
    tmp = X_tr.copy(); tmp["prix_num"] = np.exp(y_train)
    q_pm = tmp.groupby("quartier_clean")["prix_num"].median()
    q_pa = tmp.groupby("quartier_clean")["prix_num"].mean()
    q_sm = tmp.groupby("quartier_clean")["surface_num"].mean()
    l_pm = tmp.groupby("localisation_fine")["prix_num"].median()
    l_sm = tmp.groupby("localisation_fine")["surface_num"].mean()
    fp = tmp["prix_num"].median(); fs = tmp["surface_num"].mean()
    for X in [X_tr, X_te]:
        X["surface_x_quartier"]   = X["surface_num"] * X["quartier_clean"].map(q_pm).fillna(fp) / 1e6
        X["prix_m2_moy_quartier"] = X["quartier_clean"].map(q_pa).fillna(fp) / X["surface_num"]
        X["surface_relative"]     = X["surface_num"] / X["quartier_clean"].map(q_sm).fillna(fs)
        X["loc_prix_m2_median"]   = X["localisation_fine"].map(l_pm).fillna(fp) / X["surface_num"]
        X["surface_x_loc"]        = X["surface_num"] * X["localisation_fine"].map(l_pm).fillna(fp) / 1e6
        X["surface_rel_loc"]      = X["surface_num"] / X["localisation_fine"].map(l_sm).fillna(fs)
    return X_tr, X_te

X_train_enc, X_test_enc = add_groupby_features(X_train.copy(), X_test.copy())

GROUPBY_FEATURES = ["surface_x_quartier","prix_m2_moy_quartier","surface_relative",
                    "loc_prix_m2_median","surface_x_loc","surface_rel_loc"]
NUMERIC_FEATURES = NUMERIC_FEATURES_BASE + GROUPBY_FEATURES
ALL_FEATURES     = NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES



In [11]:
#  7. Pipeline + Optuna 
preprocessor = ColumnTransformer([
    ("num", StandardScaler(),                                              NUMERIC_FEATURES),
    ("bin", "passthrough",                                                 BINARY_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),   CATEGORICAL_FEATURES),
], remainder="drop")
preprocessor.fit(X_train_enc[ALL_FEATURES])

def objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 500, 3000),
        learning_rate    = trial.suggest_float("learning_rate", 0.005, 0.08, log=True),
        max_depth        = trial.suggest_int("max_depth", 3, 8),
        subsample        = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight = trial.suggest_int("min_child_weight", 3, 15),
        reg_alpha        = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        gamma            = trial.suggest_float("gamma", 0, 5),
    )
    pipe = Pipeline([("pre", preprocessor),
                     ("model", XGBRegressor(**params, random_state=42, n_jobs=-1))])
    return cross_val_score(pipe, X_train_enc[ALL_FEATURES], y_train,
                           cv=5, scoring="r2", n_jobs=-1).mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=150, show_progress_bar=True)
print(f"\n✅ Meilleur R² CV : {study.best_value:.4f}")



Best trial: 147. Best value: 0.686638: 100%|██████████| 150/150 [02:23<00:00,  1.04it/s]


✅ Meilleur R² CV : 0.6866


In [12]:
#  8. Modèle final 
pipeline_final = Pipeline([("pre", preprocessor),
                            ("model", XGBRegressor(**study.best_params, random_state=42, n_jobs=-1))])
pipeline_final.fit(X_train_enc[ALL_FEATURES], y_train)

y_pred = np.exp(pipeline_final.predict(X_test_enc[ALL_FEATURES]))
y_true = np.exp(y_test.values)

r2   = r2_score(y_true, y_pred)
mae  = mean_absolute_error(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
cv_r2 = cross_val_score(pipeline_final, X_train_enc[ALL_FEATURES], y_train,
                        cv=KFold(5, shuffle=True, random_state=42), scoring="r2")

print(f"\n{'═'*45}")
print(f"  R²   : {r2:.4f}")
print(f"  MAE  : {mae:,.0f} MAD")
print(f"  MAPE : {mape:.2f} %")
print(f"  CV R²: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"{'═'*45}")


═════════════════════════════════════════════
  R²   : 0.6508
  MAE  : 1,900,319 MAD
  MAPE : 26.02 %
  CV R²: 0.6824 ± 0.0287
═════════════════════════════════════════════


In [13]:
import joblib, json, os
from datetime import datetime

os.makedirs("../../model_training/models", exist_ok=True)

joblib.dump(pipeline_final, "../../model_training/models/xgb_villa_vente.pkl")

metadata = {
    "date"             : datetime.now().strftime("%Y-%m-%d"),
    "modele"           : "XGBRegressor",
    "type_bien"        : "villa_vente",
    "n_lignes_train"   : len(X_train_enc),
    "n_lignes_test"    : len(X_test_enc),
    "n_features"       : len(ALL_FEATURES),
    "features"         : ALL_FEATURES,
    "metriques_test"   : {"R2":round(r2,4),"MAE":round(mae,0),"MAPE":round(mape,2)},
    "hyperparametres"  : study.best_params,
    "plafond_estime"   : 0.70,
    "limite"           : "surface_terrain disponible pour 3% des annonces seulement. "
                         "35% de variance non observable (finitions, état, vue réelle).",
    "ameliorations"    : ["Scrapper surface_terrain systématiquement",
                          "API cadastre DGI", "CNN sur photos d'annonces"],
}
with open("../../model_training/models/xgb_villa_vente_metadata.json","w",encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(" Modèle final sauvegardé")
print(f"  Villas       : R²=0.65, MAPE=26%   indicatif")

 Modèle final sauvegardé
  Villas       : R²=0.65, MAPE=26%   indicatif
